In [ ]:
class Category(models.Model):
    name = models.CharField(max_length=255)
    description = models.TextField()

    def __str__(self):
        return self.name

class CategorySerializer(serializers.ModelSerializer):
    class Meta:
        model = Category
        fields = "__all__"

class CategoryViewSet(viewsets.ModelViewSet):
    queryset = Category.objects.all()
    serializer_class = CategorySerializer 

In [ ]:
fixtures
conftest.py
pytest -n 4

In [ ]:
Benefits Of unit testing

1. Early Bug detection

✅ Why use Mock?
Isolation
    You don’t want your unit test to depend on:
    Database calls
    APIs
    File systems
    External services

In [ ]:
# tests/test_category_views.py
from rest_framework.test import APITestCase
from django.urls import reverse
from rest_framework import status
from app.models import Category


class CategoryViewTestCase(APITestCase):

    def setUp(self):
        self.category = Category.objects.create(name="Books", description="All kinds of books")

    def test_create_category_success(self):
        url = reverse("category-list")
        data = {"name": "Electronics", "description": "All gadgets"}
        response = self.client.post(url, data, format="json")

        self.assertEqual(response.status_code, status.HTTP_201_CREATED)
        self.assertEqual(response.data["name"], "Electronics")

    def test_create_category_duplicate_name(self):
        url = reverse("category-list")
        data = {"name": "Books", "description": "Duplicate entry"}
        response = self.client.post(url, data, format="json")

        self.assertEqual(response.status_code, status.HTTP_400_BAD_REQUEST)

    def test_list_categories(self):
        url = reverse("category-list")
        response = self.client.get(url)

        self.assertEqual(response.status_code, status.HTTP_200_OK)
        self.assertGreaterEqual(len(response.data), 1)
        self.assertEqual(response.data[0]["name"], "Books")

    def test_retrieve_category(self):
        url = reverse("category-detail", args=[self.category.id])
        response = self.client.get(url)

        self.assertEqual(response.status_code, status.HTTP_200_OK)
        self.assertEqual(response.data["name"], "Books")

    def test_update_category(self):
        url = reverse("category-detail", args=[self.category.id])
        data = {"name": "Updated Books", "description": "Updated description"}
        response = self.client.put(url, data, format="json")

        self.assertEqual(response.status_code, status.HTTP_200_OK)
        self.assertEqual(response.data["name"], "Updated Books")

    def test_partial_update_category(self):
        url = reverse("category-detail", args=[self.category.id])
        data = {"description": "Partially updated description"}
        response = self.client.patch(url, data, format="json")

        self.assertEqual(response.status_code, status.HTTP_200_OK)
        self.assertEqual(response.data["description"], "Partially updated description")

    def test_delete_category_success(self):
        url = reverse("category-detail", args=[self.category.id])
        response = self.client.delete(url)

        self.assertEqual(response.status_code, status.HTTP_204_NO_CONTENT)
        self.assertEqual(Category.objects.count(), 0)

    def test_delete_category_not_found(self):
        url = reverse("category-detail", args=[999])  # Non-existent ID
        response = self.client.delete(url)

        self.assertEqual(response.status_code, status.HTTP_404_NOT_FOUND)


In [ ]:
🔹 Mock Test for List API

# tests/test_category_mock.py
from unittest.mock import patch, MagicMock
from rest_framework.test import APITestCase
from django.urls import reverse
from rest_framework import status
from app.models import Category


class CategoryApiMockTestCase(APITestCase):

    @patch("app.views.Category.objects.all")  # Mock ORM call
    def test_list_categories_mock(self, mock_all):
        # Arrange
        mock_category = MagicMock(spec=Category, id=1, name="Mocked", description="Fake data")
        mock_all.return_value = [mock_category]

        url = reverse("category-list")

        # Act
        response = self.client.get(url)

        # Assert
        self.assertEqual(response.status_code, status.HTTP_200_OK)
        self.assertEqual(response.data[0]["name"], "Mocked")
        mock_all.assert_called_once()
